# 🏨 Hotel Guests Dataset — End-to-End Data Analysis
**Objective:** Perform complete exploratory data analysis on hotel guests booking data including cleaning, feature engineering, and visualizations.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries imported successfully!')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('hotel_guests_dataset.csv')
df.head()

## 3. Initial Exploration

In [ ]:
# Shape of dataset
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

In [ ]:
# Column data types and non-null counts
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Check for duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

## 4. Data Cleaning

In [ ]:
# Drop the unnamed index column
df.drop(columns=['Unnamed: 0'], inplace=True)

# Convert date columns to datetime
df['checkin_date'] = pd.to_datetime(df['checkin_date'], format='%d %b %Y')
df['checkout_date'] = pd.to_datetime(df['checkout_date'], format='%d %b %Y')

print('Date columns converted.')
df[['checkin_date', 'checkout_date']].dtypes

In [ ]:
# Handle missing checkout_date: fill with checkin + median stay duration
df['stay_duration'] = (df['checkout_date'] - df['checkin_date']).dt.days
median_stay = df['stay_duration'].median()
print(f'Median stay duration: {median_stay} days')

# Fill missing checkout_date
mask_checkout = df['checkout_date'].isnull()
df.loc[mask_checkout, 'checkout_date'] = df.loc[mask_checkout, 'checkin_date'] + pd.Timedelta(days=median_stay)

# Recalculate stay duration after fill
df['stay_duration'] = (df['checkout_date'] - df['checkin_date']).dt.days
print(f'Missing checkout_date after fill: {df["checkout_date"].isnull().sum()}')

In [ ]:
# Handle missing amenities_fee: fill with median grouped by room_type
df['amenities_fee'] = df.groupby('room_type')['amenities_fee'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Missing amenities_fee after fill: {df["amenities_fee"].isnull().sum()}')

In [ ]:
# Verify no missing values remain
df.isnull().sum()

## 5. Feature Engineering

In [ ]:
# Extract checkin month, year, day of week
df['checkin_year']  = df['checkin_date'].dt.year
df['checkin_month'] = df['checkin_date'].dt.month
df['checkin_month_name'] = df['checkin_date'].dt.strftime('%b')
df['checkin_dayofweek'] = df['checkin_date'].dt.day_name()

# Total bill = room_rate * stay_duration + amenities_fee
df['total_bill'] = (df['room_rate'] * df['stay_duration']) + df['amenities_fee']

# Categorize stay duration
def categorize_stay(days):
    if days <= 3:
        return 'Short (1-3 days)'
    elif days <= 7:
        return 'Medium (4-7 days)'
    elif days <= 14:
        return 'Long (8-14 days)'
    else:
        return 'Extended (15+ days)'

df['stay_category'] = df['stay_duration'].apply(categorize_stay)

# Categorize room_rate into bins
edges = [df['room_rate'].min(), df['room_rate'].quantile(0.25),
         df['room_rate'].quantile(0.5), df['room_rate'].quantile(0.75),
         df['room_rate'].max()]
labels = ['Budget', 'Economy', 'Standard', 'Premium']
df['rate_category'] = pd.cut(df['room_rate'], bins=edges, labels=labels, duplicates='drop')

df.head()

## 6. Univariate Analysis

In [ ]:
# Room type distribution
ax = sns.catplot(y='room_type', data=df, kind='count',
                 order=df['room_type'].value_counts().index, color='#4287f5', height=4, aspect=2)
plt.title('Room Type Distribution')
plt.xlabel('Count')
plt.ylabel('Room Type')
plt.show()

In [ ]:
# Rewards members vs non-members
ax = sns.catplot(x='has_rewards', data=df, kind='count', palette=['#f54242', '#4287f5'], height=4, aspect=1.5)
plt.title('Rewards Membership Distribution')
plt.xlabel('Has Rewards')
plt.ylabel('Count')
plt.show()

In [ ]:
# Room rate distribution
plt.figure(figsize=(10, 4))
sns.histplot(df['room_rate'], bins=30, kde=True, color='#4287f5')
plt.title('Room Rate Distribution')
plt.xlabel('Room Rate ($)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Amenities fee distribution
plt.figure(figsize=(10, 4))
sns.histplot(df['amenities_fee'], bins=30, kde=True, color='#f5a742')
plt.title('Amenities Fee Distribution')
plt.xlabel('Amenities Fee ($)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Stay duration distribution
plt.figure(figsize=(10, 4))
sns.histplot(df['stay_duration'], bins=20, kde=True, color='#42f587')
plt.title('Stay Duration Distribution (Days)')
plt.xlabel('Stay Duration (Days)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Stay category distribution
stay_order = ['Short (1-3 days)', 'Medium (4-7 days)', 'Long (8-14 days)', 'Extended (15+ days)']
sns.catplot(y='stay_category', data=df, kind='count', order=stay_order, color='#9b42f5', height=4, aspect=2)
plt.title('Stay Category Distribution')
plt.xlabel('Count')
plt.show()

## 7. Time-Series Analysis

In [ ]:
# Bookings by year
yearly = df['checkin_year'].value_counts().sort_index()
plt.figure(figsize=(10, 4))
sns.barplot(x=yearly.index, y=yearly.values, palette='Blues_d')
plt.title('Bookings by Year')
plt.xlabel('Year')
plt.ylabel('Number of Bookings')
plt.show()

In [ ]:
# Bookings by month
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df['checkin_month_name'].value_counts().reindex(month_order)
plt.figure(figsize=(12, 4))
sns.barplot(x=monthly.index, y=monthly.values, palette='coolwarm')
plt.title('Bookings by Month')
plt.xlabel('Month')
plt.ylabel('Number of Bookings')
plt.show()

In [ ]:
# Bookings by day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df['checkin_dayofweek'].value_counts().reindex(dow_order)
plt.figure(figsize=(10, 4))
sns.barplot(x=dow.index, y=dow.values, palette='viridis')
plt.title('Bookings by Day of Week')
plt.xlabel('Day')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

## 8. Bivariate & Multivariate Analysis

In [ ]:
# Average room rate by room type
plt.figure(figsize=(8, 4))
sns.barplot(x='room_type', y='room_rate', data=df,
            order=['BASIC', 'DELUXE', 'SUITE'], palette='Blues_d')
plt.title('Average Room Rate by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Avg Room Rate ($)')
plt.show()

In [ ]:
# Room rate by room type — boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x='room_type', y='room_rate', data=df,
            order=['BASIC', 'DELUXE', 'SUITE'], palette='Set2')
plt.title('Room Rate Distribution by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Room Rate ($)')
plt.show()

In [ ]:
# Room rate vs has_rewards
plt.figure(figsize=(8, 4))
sns.boxplot(x='has_rewards', y='room_rate', data=df, palette=['#f54242', '#4287f5'])
plt.title('Room Rate: Rewards Members vs Non-Members')
plt.xlabel('Has Rewards')
plt.ylabel('Room Rate ($)')
plt.show()

In [ ]:
# Total bill by room type
plt.figure(figsize=(8, 4))
sns.barplot(x='room_type', y='total_bill', data=df,
            order=['BASIC', 'DELUXE', 'SUITE'], palette='Oranges_d')
plt.title('Average Total Bill by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Avg Total Bill ($)')
plt.show()

In [ ]:
# Stay duration vs room type
plt.figure(figsize=(8, 4))
sns.boxplot(x='room_type', y='stay_duration', data=df,
            order=['BASIC', 'DELUXE', 'SUITE'], palette='Pastel1')
plt.title('Stay Duration by Room Type')
plt.xlabel('Room Type')
plt.ylabel('Stay Duration (Days)')
plt.show()

In [ ]:
# Room type by rewards membership — grouped count
plt.figure(figsize=(8, 4))
crosstab = pd.crosstab(df['room_type'], df['has_rewards'])
crosstab.plot(kind='bar', color=['#f54242', '#4287f5'], edgecolor='white')
plt.title('Room Type by Rewards Membership')
plt.xlabel('Room Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Has Rewards')
plt.show()

In [ ]:
# Monthly revenue trend (total_bill)
df_sorted = df.copy()
df_sorted['month_year'] = df_sorted['checkin_date'].dt.to_period('M')
monthly_rev = df_sorted.groupby('month_year')['total_bill'].sum().reset_index()
monthly_rev['month_year'] = monthly_rev['month_year'].astype(str)

plt.figure(figsize=(14, 5))
plt.plot(monthly_rev['month_year'], monthly_rev['total_bill'], marker='o', color='#4287f5', linewidth=2)
plt.title('Monthly Revenue Trend (Total Bill)')
plt.xlabel('Month')
plt.ylabel('Total Revenue ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Correlation Analysis

In [ ]:
# Correlation heatmap for numeric columns
numeric_cols = ['room_rate', 'amenities_fee', 'stay_duration', 'total_bill']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Scatter: room_rate vs total_bill colored by room_type
plt.figure(figsize=(10, 5))
colors = {'BASIC': '#4287f5', 'DELUXE': '#f5a742', 'SUITE': '#f54242'}
for rtype, grp in df.groupby('room_type'):
    plt.scatter(grp['room_rate'], grp['total_bill'],
                label=rtype, alpha=0.5, color=colors[rtype], s=20)
plt.title('Room Rate vs Total Bill by Room Type')
plt.xlabel('Room Rate ($)')
plt.ylabel('Total Bill ($)')
plt.legend()
plt.show()

## 10. Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['room_rate', 'amenities_fee', 'total_bill']):
    sns.boxplot(y=df[col], ax=ax, color='#4287f5')
    ax.set_title(f'{col} — Outliers')
plt.tight_layout()
plt.show()

# IQR-based outlier count
for col in ['room_rate', 'amenities_fee', 'stay_duration', 'total_bill']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f'{col}: {len(outliers)} outliers ({100*len(outliers)/len(df):.1f}%)')

## 11. Key Insights Summary

In [ ]:
print('=' * 55)
print('           HOTEL GUESTS — KEY INSIGHTS')
print('=' * 55)
print(f'Total Records               : {len(df):,}')
print(f'Total Revenue (est.)        : ${df["total_bill"].sum():,.2f}')
print(f'Avg Room Rate               : ${df["room_rate"].mean():.2f}')
print(f'Avg Stay Duration           : {df["stay_duration"].mean():.1f} days')
print(f'Avg Total Bill              : ${df["total_bill"].mean():.2f}')
print(f'Rewards Members             : {df["has_rewards"].sum()} ({100*df["has_rewards"].mean():.1f}%)')
print(f'Most Popular Room Type      : {df["room_type"].mode()[0]}')
print(f'Most Common Stay Category   : {df["stay_category"].mode()[0]}')
print(f'Busiest Month               : {df["checkin_month_name"].mode()[0]}')
print(f'Top Revenue Room Type       : {df.groupby("room_type")["total_bill"].sum().idxmax()}')
print('=' * 55)